# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FarihaKarim12/machine-learning/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/FarihaKarim12/machine-learning"
REPO_DIR = "machine-learning"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # running locally — just make sure you're at the repo root
    pass

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## My Lane: Ranking Signal Analysis

I'm choosing Lane 1 — Ranking Signal Analysis

**Why:** Notebook 01 already surfaced three real findings from the starter dataset (search
volume barely predicts impressions, CTR collapses sharply by position tier, word count is not
a strong lever for decline vs. growth). Each of those is exactly the kind of question this lane
is built around: which safe, observable signals actually associate with visibility, clicks, or
movement — and which commonly-assumed levers don't hold up. This lane lets me dig deeper into
that thread with EDA, correlations, and simple models, rather than jumping straight to a
black-box scoring pipeline (Lane 2) before I understand which signals are worth scoring on
in the first place.

I may revisit this by Week 4 if the evidence points me toward Lane 2 (Refresh/Opportunity
Scoring) as a natural next step, since a strong signal audit is a prerequisite for a good
scoring model anyway.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## The Question

**Research question:** Which safe, observable content and search signals are actually
associated with a page's visibility, clicks, or performance movement — and which commonly
assumed levers (like content length or keyword search volume) are not?

**Unit of analysis:** One page (content_id) in the anonymized starter dataset, with metrics
aggregated over a 90-day window.

**Output:** A signal report — which features correlate meaningfully with visibility/CTR/trend,
with effect sizes, not just "is there a link."

**The decision this improves:** Which signals a content team should prioritize checking first
when reviewing a page, and which "common wisdom" levers (e.g. "make it longer") they should
stop over-indexing on.

**The action someone could take:** A content reviewer or SEO strategist could use this to
build a shorter, evidence-backed checklist for page reviews — e.g. checking position tier and
CTR gap before checking word count, since the data suggests position/CTR carries more signal
than length.

**Cost of a wrong call:** If the signal report is wrong (e.g. claims word count matters when
it doesn't), reviewers waste limited time and effort padding out content that was never the
actual problem, while the pages with real position/CTR issues go unreviewed. The cost isn't
catastrophic per page, but it compounds — it's wasted reviewer capacity across the whole queue.

**Why data/ML helps here (not just a model):** The core value of this lane isn't a prediction
— it's correcting mistaken intuitions with evidence. A correlation check and a grouped
comparison can overturn a belief that's been guiding decisions without ever being tested. That's
useful even before any model gets built.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")

# Number 1: search_volume barely predicts real impressions
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation between search_volume and impressions_90d: {corr:.3f}")
print("Near zero -> a commonly assumed lever (target high-search-volume keywords) doesn't")
print("track with what actually shows up in search.")

# Number 2: CTR collapses sharply by position tier (magnitude, not just direction)
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print(ctr_by_pos.round(4).to_string())
top_ctr, bottom_ctr = ctr_by_pos.iloc[0], ctr_by_pos.iloc[-1]
print(f"\nTop tier CTR is roughly {top_ctr/bottom_ctr:.1f}x the bottom tier's CTR.")
print("Position is a strong, measurable lever -- much stronger than word count (below).")

# Number 3: content length is not a strong lever for decline vs. growth
wc = df.groupby("trend_direction")["word_count"].median()
print(wc.round(0).to_string())
print("\n'down' vs 'up' pages have nearly identical median word count.")
print("This directly challenges a common assumption that longer content prevents decline.")

30000 rows, 44 columns
Correlation between search_volume and impressions_90d: 0.001
Near zero -> a commonly assumed lever (target high-search-volume keywords) doesn't
track with what actually shows up in search.
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554

Top tier CTR is roughly 6.4x the bottom tier's CTR.
Position is a strong, measurable lever -- much stronger than word count (below).
trend_direction
down      2909.0
flat      2698.0
new       2239.0
stable    2912.0
up        2848.0

'down' vs 'up' pages have nearly identical median word count.
This directly challenges a common assumption that longer content prevents decline.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:** These are observed, directional patterns from a 30,000-row anonymized
sample. Search volume shows almost no linear relationship with actual impressions in this
data. CTR is strongly tied to position tier. Word count shows little difference between
declining and growing pages. These are associations, not causes.

**What I cannot claim:** I cannot claim any of this reflects how Google's ranking algorithm
actually works — I only observe outcomes, not the mechanism. I cannot claim a page will
recover if its word count is changed, since I haven't run any experiment — only observed
correlation. I cannot generalize beyond this dataset's window and the clients represented in
it; the full warehouse or a different time period could show different patterns. I'm not
using any of FlyRank's own product decision scores (health_score, priority_score, etc.) as
either a feature or a claim — these aren't in my dataset, so there's nothing to worry about
leaking, but I also won't treat any rebuilt version of them as ground truth later on.

I'll frame every finding as "observed" or "directional," never as "proven."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.